In [1]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))
from utils.PDF2MarkdownProcessor import PDF2MarkdownProcessor
from academic_research_assistant import embed_documents, markdown_splitter
from dotenv import load_dotenv
import os
import uuid

c:\Users\youss\OneDrive\Desktop\Academic_Knowledge_&_Research_Assistant\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [10]:
# Set up ChromaDB client and create a collection
import chromadb
chroma_client = chromadb.Client()
collection = chroma_client.create_collection(name="documents")

In [3]:
# Set up the tokenizer and token limit
from transformers import AutoTokenizer

load_dotenv()

HF_TOKEN = os.getenv("HG_ACCESS_TOKEN")

tokenizer = AutoTokenizer.from_pretrained(
    "google/gemma-3-4b-it",
    token=HF_TOKEN
)

token_limit = 512

In [4]:
# Embed the documents and add them to the ChromaDB collection
pdf_path = Path.cwd().parent / "documents" / "pdfs" / "2410.01574v4.pdf"
converter = PDF2MarkdownProcessor()
embedded_document = embed_documents.embed_document(pdf_path, converter, tokenizer, token_limit)
print(f"Total chunks created: {len(embedded_document)}")

Total chunks created: 46


In [6]:
# Print the embedded document chunks for verification
for chunk in embedded_document:
    print(f"Chunk index: {chunk.chunk_index}")
    print(f"Source: {chunk.source}")
    print(f"Text snippet: {chunk.text[:50]}...")  
    print(f"Embedding vector length: {len(chunk.embedding)}")
    print(f"Embedding snippet: {chunk.embedding[:5]}...")
    print(f"Token count: {chunk.token_count}") 
    print("-" * 40)  

Chunk index: 0
Source: 2410.01574v4.pdf
Text snippet: # **Adversarial Robustness of AI-Generated Image D...
Embedding vector length: 1024
Embedding snippet: [0.01875837, -0.017534407, -0.0021773903, -0.012823156, 0.037036233]...
Token count: 385
----------------------------------------
Chunk index: 1
Source: 2410.01574v4.pdf
Text snippet: ## **1 Introduction**

The creation of synthetic v...
Embedding vector length: 1024
Embedding snippet: [-0.021400077, 0.0034286012, -0.0019403495, 0.025034532, 0.028112482]...
Token count: 312
----------------------------------------
Chunk index: 2
Source: 2410.01574v4.pdf
Text snippet: Since AI-generated images (AIGIs) are often widely...
Embedding vector length: 1024
Embedding snippet: [0.0134520065, -0.035696507, -0.00042847308, 0.007602457, 0.008198906]...
Token count: 361
----------------------------------------
Chunk index: 3
Source: 2410.01574v4.pdf
Text snippet: In this work, we address these limitations and eva...
Embedding vector length: 102

In [11]:
# Add the embedded document chunks to the ChromaDB collection
for chunk in embedded_document:
    collection.add(
        documents=[chunk.text],
        metadatas=[{"source": chunk.source, "chunk_index": chunk.chunk_index}],
        ids=[str(uuid.uuid4())],
        embeddings=[chunk.embedding]
    )

In [12]:
# Embed a query to retrieve relevant document chunks based on the question
query = "what architectural relationship appears to explain the difference in PGD transferability between the DRCT-CLIP → UnivFD attack and the DRCT-ConvB → UnivFD attack, and what evidence in the reported AUROC values supports this interpretation?"
query_embedding = embed_documents.embed_batch([query])[0]
print(f"Query embedding length: {len(query_embedding)}")

Query embedding length: 1024


In [13]:
results = collection.query(
    query_embeddings=[query_embedding],
    n_results=2
)
print(results['documents'])

[['5|59.1|36.1|44.9|24.7|\n|||UnivFD|7.5|38.5|59.1|30.4|49.0|50.6|0.0|4.5|11.7|\n||DRCT-CLIP|DRCT-ConvB|62.3|68.5|62.5|52.8|50.4|51.3|14.7|21.2|14.1|\n|**PGD**||Corvi|63.6|71.7|63.4|53.0|50.6|51.3|18.8|26.7|14.1|\n|||UnivFD|58.5|53.6|47.0|50.5|49.9|48.1|14.0|9.3|4.7|\n||DRCT-ConvB|DRCT-CLIP|16.4|45.1|46.9|44.5|47.7|48.6|0.8|4.5|3.5|\n|||Corvi|0.7|5.5|42.8|7.6|14.4|47.0|0.0|0.0|3.5|\n|||UnivFD|63.3|54.0|49.2|58.2|50.2|50.0|29.9|16.3|6.5|\n||Corvi|DRCT-CLIP|60.9|57.9|50.9|56.9|50.1|50.0|27.2|16.4|8.1|\n|||DRCT-ConvB|76.4|67.5|53.6|60.3|50.0|50.0|38.0|24.1|8.3|\n||Average|–|48.8|57.6|56.5|48', '**Attack Comparison.** Table 1 presents our robustness analysis averaged across all datasets, reporting AUROC alongside Accuracy and TPR@5%FPR. We evaluate the attacks and their transferability across different degradation levels, where “None” refers to attacks without degradation. “LTO” refers to “LeaveTarget-Out”: For each target detector, the ensemble attack is optimized on all other detectors a